# 02 — Chunking Tests

Experiment with chunk sizes, overlap, and splitting strategies.

In [5]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

# List all collections
collections = client.list_collections()

print(f"Found {len(collections)} collection(s):\n")
for c in collections:
    print(f"  Name: '{c.name}'")
    print(f"  Items: {c.count()}")
    print(f"  Metadata: {c.metadata}")
    print()

Found 0 collection(s):



In [6]:
from pathlib import Path

db_path = Path("./chroma_db")
print(f"Database folder: {db_path.absolute()}")
print(f"Exists: {db_path.exists()}")

if db_path.exists():
    print("\nContents:")
    for item in db_path.iterdir():
        if item.is_file():
            print(f"  📄 {item.name} ({item.stat().st_size:,} bytes)")
        else:
            print(f"  📁 {item.name}/")
            for sub in item.iterdir():
                print(f"    {sub.name}")

Database folder: e:\P1_JOB\AI projects\financial_risk_rag\notebooks\chroma_db
Exists: True

Contents:
  📄 chroma.sqlite3 (188,416 bytes)


In [7]:
from pathlib import Path

# Search in common locations
project_root = Path("E:/P1_JOB/AI projects/financial_risk_rag")

print("Searching for ChromaDB folders...\n")

candidates = list(project_root.rglob("chroma.sqlite3"))

for db_file in candidates:
    folder = db_file.parent
    size_kb = db_file.stat().st_size / 1024
    print(f"Found: {db_file}")
    print(f"  Size: {size_kb:.0f} KB")
    print(f"  Folder: {folder}")
    print()

Searching for ChromaDB folders...

Found: E:\P1_JOB\AI projects\financial_risk_rag\chroma_db\chroma.sqlite3
  Size: 12988 KB
  Folder: E:\P1_JOB\AI projects\financial_risk_rag\chroma_db

Found: E:\P1_JOB\AI projects\financial_risk_rag\notebooks\chroma_db\chroma.sqlite3
  Size: 184 KB
  Folder: E:\P1_JOB\AI projects\financial_risk_rag\notebooks\chroma_db



In [10]:
import chromadb

# Use the ABSOLUTE path to your real ChromaDB
db_path = "E:/P1_JOB/AI projects/financial_risk_rag/chroma_db"  # ← update based on Step 1

client = chromadb.PersistentClient(path=db_path)

# List collections
collections = client.list_collections()
print(f"Found {len(collections)} collection(s):\n")
for c in collections:
    print(f"  Name: '{c.name}'")
    print(f"  Items: {c.count()}")

Found 1 collection(s):

  Name: 'financial_risk_10k'
  Items: 1401


In [9]:
import shutil
from pathlib import Path

empty_db = Path("E:/P1_JOB/AI projects/financial_risk_rag/notebooks/chroma_db")
if empty_db.exists():
    shutil.rmtree(empty_db)
    print(f"Deleted empty database: {empty_db}")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'E:\\P1_JOB\\AI projects\\financial_risk_rag\\notebooks\\chroma_db\\chroma.sqlite3'

In [11]:
"""
Inspect chunks for a single source document by querying ChromaDB.

Self-contained — connects to ChromaDB and pulls chunks for the specified source.
No external chunks_list needed.
"""

import random
import chromadb
from pathlib import Path

# Define project root once at the top of your notebook
PROJECT_ROOT = Path("E:/P1_JOB/AI projects/financial_risk_rag")
CHROMA_DB_PATH = str(PROJECT_ROOT / "chroma_db")  # adjust based on where yours actually is


def inspect_chunks(source_filename, collection_name="financial_risk_10k", db_path=CHROMA_DB_PATH):
    """
    Inspect chunks for a single source document.
    
    Args:
        source_filename: Filename to inspect (e.g., 'AXP_..._risk_factors.txt')
        collection_name: ChromaDB collection name
        db_path: Path to ChromaDB persistent storage
    """
    # Connect to ChromaDB
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_collection(collection_name)
    
    # Get all chunks from this source (using metadata filter)
    results = collection.get(
        where={"source": source_filename},
        include=['documents', 'metadatas']
    )
    
    chunks = results['documents']
    metadatas = results['metadatas']
    
    if len(chunks) == 0:
        print(f"⚠️ No chunks found for source: {source_filename}")
        print(f"\nAvailable sources in collection:")
        all_data = collection.get(include=['metadatas'])
        unique_sources = set(m.get('source') for m in all_data['metadatas'] if 'source' in m)
        for s in sorted(unique_sources):
            print(f"  - {s}")
        return
    
    print("=" * 60)
    print(f"INSPECTING CHUNKS: {source_filename}")
    print(f"Total chunks: {len(chunks)}")
    print("=" * 60)
    
    # Pair chunks with metadata, sort by chunk_index if available
    chunk_pairs = list(zip(chunks, metadatas))
    if 'chunk_index' in metadatas[0]:
        chunk_pairs.sort(key=lambda x: x[1].get('chunk_index', 0))
    
    # Sample 1: First chunk
    print(f"\n--- FIRST CHUNK ---")
    text, meta = chunk_pairs[0]
    print(f"Length: {len(text)} chars")
    print(f"Metadata: {meta}")
    print(f"\nContent (first 400 chars):")
    print(text[:400])
    
    # Sample 2: Random middle chunk
    if len(chunk_pairs) > 3:
        mid_start = len(chunk_pairs) // 3
        mid_end = 2 * len(chunk_pairs) // 3
        mid_text, mid_meta = random.choice(chunk_pairs[mid_start:mid_end])
        print(f"\n--- RANDOM MIDDLE CHUNK ---")
        print(f"Length: {len(mid_text)} chars")
        print(f"Metadata: {mid_meta}")
        print(f"\nContent (first 400 chars):")
        print(mid_text[:400])
    
    # Sample 3: Last chunk
    print(f"\n--- LAST CHUNK ---")
    text, meta = chunk_pairs[-1]
    print(f"Length: {len(text)} chars")
    print(f"Metadata: {meta}")
    print(f"\nContent (first 400 chars):")
    print(text[:400])
    
    # Sample 4: Three smallest chunks (often reveal problems)
    sorted_by_size = sorted(chunk_pairs, key=lambda x: len(x[0]))
    print(f"\n--- THREE SMALLEST CHUNKS (often problematic) ---")
    for text, meta in sorted_by_size[:3]:
        print(f"\nLength: {len(text)} chars, Chunk index: {meta.get('chunk_index', '?')}")
        print(f"Content: {repr(text[:200])}")
    
    # Sample 5: Three largest chunks
    print(f"\n--- THREE LARGEST CHUNKS ---")
    for text, meta in sorted_by_size[-3:]:
        print(f"\nLength: {len(text)} chars, Chunk index: {meta.get('chunk_index', '?')}")
        print(f"Content (first 200 chars): {text[:200]}")
    
    # Size statistics
    sizes = [len(t) for t, m in chunk_pairs]
    print(f"\n--- SIZE STATISTICS ---")
    print(f"Average chunk size: {sum(sizes) / len(sizes):.0f} chars")
    print(f"Min: {min(sizes)} chars")
    print(f"Max: {max(sizes)} chars")
    chunks_under_300 = sum(1 for s in sizes if s < 300)
    print(f"Chunks under 300 chars: {chunks_under_300} ({chunks_under_300/len(sizes)*100:.1f}%)")


# Usage — try with one of your actual files
inspect_chunks('AXP_0000004962-26-000080_risk_factors.txt')

INSPECTING CHUNKS: AXP_0000004962-26-000080_risk_factors.txt
Total chunks: 142

--- FIRST CHUNK ---
Length: 1184 chars
Metadata: {'char_end': 1184, 'chunk_index': 0, 'source': 'AXP_0000004962-26-000080_risk_factors.txt', 'file_type': 'risk_factors', 'filing_year': 2026, 'item': '1A', 'ticker': 'AXP', 'company': 'American Express', 'char_start': 0}

Content (first 400 chars):
ITEM 1A.    RISK FACTORS
This section highlights certain risks that could affect us and our businesses, broadly categorized in accordance with the risk types identified in our risk governance framework: “Strategic and Reputational Risks,” “Operational and Compliance Risks” and “Credit, Market and Liquidity Risks.” You should carefully consider each of the following risks and all of the other infor

--- RANDOM MIDDLE CHUNK ---
Length: 904 chars
Metadata: {'ticker': 'AXP', 'source': 'AXP_0000004962-26-000080_risk_factors.txt', 'filing_year': 2026, 'item': '1A', 'company': 'American Express', 'chunk_index': 49, 'char_

In [4]:
"""
Setup — required before any 'from src.* import ...' works.
"""
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Find project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# CRITICAL: Add project root to Python's import path
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env
load_dotenv(PROJECT_ROOT / ".env", override=True)
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY not loaded"

# Build retriever (loads ChromaDB + BM25 — takes ~10 sec)
from src.retrieval import HybridRetriever

retriever = HybridRetriever(
    chroma_path=str(PROJECT_ROOT / "chroma_db"),
    collection_name="financial_risk_10k",
)
print(f"\n✓ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✓ Retriever ready with {len(retriever.chunk_texts)} chunks")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading chunks from collection 'financial_risk_10k'...
Building BM25 index over 1401 chunks...
✓ HybridRetriever ready: 1401 chunks

✓ PROJECT_ROOT: e:\P1_JOB\AI projects\financial_risk_rag
✓ Retriever ready with 1401 chunks


In [6]:
"""
Spot-check helper: pull chunks for a question and inspect them.

Use this to verify that my reference answer actually reflects 
what's in the corpus for that question.
"""
from src.generation import Generator
from src.pipeline import RAGPipeline

# Reuse the retriever you already built
generator = Generator()
pipeline = RAGPipeline(retriever=retriever, generator=generator, top_k=5)


def inspect_question(question: str, top_k: int = 5, show_full: bool = False):
    """Show the chunks that would be retrieved for this question."""
    print("=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)
    
    chunks = retriever.hybrid_search(question, top_k=top_k, candidate_pool=25)
    
    for i, chunk in enumerate(chunks, 1):
        ticker = chunk.metadata.get("ticker", "?")
        print(f"\n[Chunk {i}] Ticker: {ticker}  |  RRF score: {chunk.score:.4f}")
        print(f"          Dense rank: {chunk.dense_rank}  |  BM25 rank: {chunk.bm25_rank}")
        print("-" * 60)
        if show_full:
            print(chunk.text)
        else:
            # Show first 400 chars + last 200 chars
            text = chunk.text
            if len(text) > 600:
                print(text[:400])
                print("    [...]")
                print(text[-200:])
            else:
                print(text)
    print()


# Try it on one comparison question
inspect_question(
    "How do JPM and GS describe their stress-testing and capital requirements differently?"
)

QUESTION: How do JPM and GS describe their stress-testing and capital requirements differently?

[Chunk 1] Ticker: COF  |  RRF score: 0.0331
          Dense rank: 0  |  BM25 rank: 1
------------------------------------------------------------
We consider various factors in the management of capital, including the impact of both internal and supervisory stress scenarios on our capital levels as determined by our internal modeling and the Federal Reserve’s estimation of losses in supervisory stress scenarios that are used to annually set our stress capital buffer requirement. Although the Federal Reserve has issued the Stress Testing Tra
    [...]
r other factors related to our capital management process may reflect a lower capacity to return capital to stockholders than that indicated by the projections released in the stress testing processes

[Chunk 2] Ticker: BAC  |  RRF score: 0.0321
          Dense rank: 5  |  BM25 rank: 0
-----------------------------------------------------------

In [7]:
def view_chunk_by_index(idx: int):
    """View a chunk by its position in the corpus."""
    text = retriever.chunk_texts[idx]
    meta = retriever.chunk_metas[idx]
    chunk_id = retriever.chunk_ids[idx]
    
    print(f"Chunk ID: {chunk_id}")
    print(f"Metadata: {meta}")
    print(f"Length: {len(text)} chars, {len(text.split())} words")
    print("-" * 60)
    print(text)


def view_chunks_for_ticker(ticker: str, limit: int = 3):
    """Look at the first N chunks for a specific company."""
    matches = [
        (i, retriever.chunk_metas[i], retriever.chunk_texts[i])
        for i in range(len(retriever.chunk_metas))
        if retriever.chunk_metas[i].get("ticker") == ticker
    ]
    print(f"Showing {min(limit, len(matches))} of {len(matches)} chunks for {ticker}\n")
    
    for idx, meta, text in matches[:limit]:
        print(f"[idx={idx}] {meta}")
        print(text[:500])
        print("-" * 60)


# Examples:
view_chunk_by_index(0)
view_chunks_for_ticker("JPM", limit=2)

Chunk ID: AXP_0000004962-26-000080_risk_factors_0
Metadata: {'company': 'American Express', 'source': 'AXP_0000004962-26-000080_risk_factors.txt', 'ticker': 'AXP', 'char_start': 0, 'char_end': 1184, 'chunk_index': 0, 'file_type': 'risk_factors', 'filing_year': 2026, 'item': '1A'}
Length: 1184 chars, 176 words
------------------------------------------------------------
ITEM 1A.    RISK FACTORS
This section highlights certain risks that could affect us and our businesses, broadly categorized in accordance with the risk types identified in our risk governance framework: “Strategic and Reputational Risks,” “Operational and Compliance Risks” and “Credit, Market and Liquidity Risks.” You should carefully consider each of the following risks and all of the other information set forth in this Annual Report on Form 10-K, including in “Risk Management” under “MD&A,” which describes our approach to identifying, monitoring and managing the risks we assume in conducting our businesses and provides

In [8]:
"""
Investigate why COF has 2x the chunk count of the median.

Possible explanations (in order of likelihood):
1. COF's Item 1A really is twice as long as the others
2. Chunking produced unusually small chunks for COF (some edge case)
3. Extraction included extra text (boilerplate, repeated sections)
"""
import statistics

# Compute per-ticker statistics
ticker_stats = {}
for ticker in sorted(set(m.get("ticker") for m in retriever.chunk_metas)):
    sizes = [
        len(retriever.chunk_texts[i])
        for i in range(len(retriever.chunk_metas))
        if retriever.chunk_metas[i].get("ticker") == ticker
    ]
    ticker_stats[ticker] = {
        "n_chunks": len(sizes),
        "total_chars": sum(sizes),
        "mean_chunk_size": statistics.mean(sizes),
        "median_chunk_size": statistics.median(sizes),
        "min_chunk_size": min(sizes),
        "max_chunk_size": max(sizes),
    }

print(f"{'Ticker':<8} {'Chunks':<8} {'TotalChars':<12} {'MeanSize':<10} {'MedSize':<10} {'MinSize':<10}")
print("-" * 60)
for t, s in ticker_stats.items():
    print(
        f"{t:<8} {s['n_chunks']:<8} {s['total_chars']:<12,} "
        f"{s['mean_chunk_size']:<10.0f} {s['median_chunk_size']:<10.0f} "
        f"{s['min_chunk_size']:<10}"
    )

Ticker   Chunks   TotalChars   MeanSize   MedSize    MinSize   
------------------------------------------------------------
AXP      142      113,543      800        908        69        
BAC      166      134,828      812        873        91        
BLK      143      111,770      782        891        55        
COF      280      231,892      828        886        27        
GS       170      140,466      826        910        98        
JPM      125      117,966      944        1031       14        
MA       101      83,782       830        945        6         
PYPL     174      137,415      790        868        20        
V        100      82,625       826        924        74        
